# Method 2 — iLTM-Inspired Vanilla-GBDT + Random-ReLU + MLP + Soft Retrieval

This notebook implements the frozen Method 2 architecture agreed for the Kickstarter regression experiment.

**Pipeline:** Vanilla/under-tuned XGBoost → leaf indices → one-hot leaf embedding Γ(x) → concatenate with robustly preprocessed original features → random feature expansion + ReLU → PCA → normalization → MLP → penultimate representation → soft k-NN retrieval → validation-selected α → final test prediction.

The implementation is deliberately **not** a tuned-XGBoost feature extractor and does not use the previous tuned XGBoost checkpoint.

The architecture follows the paper's embedding and retrieval structure: the paper defines one-hot GBDT leaf embeddings, concatenation with preprocessed features, random feature expansion followed by PCA and feature-wise normalization, and retrieval from the MLP penultimate representation using cosine similarity and temperature-scaled softmax weights. citeturn0view0

This notebook is an **iLTM-inspired single-dataset regression adaptation**, not the complete meta-trained iLTM system.

# 1. Imports, Paths, and Reproducibility

All stochastic components use seed 42. The uploaded datasets are used directly, and the checkpoint directory is used only for saving this experiment's artifacts.

In [1]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_squared_log_error
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

import xgboost as xgb

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

TRAIN_PATH = Path(r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_train.csv")
TEST_PATH = Path(r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_test.csv")

CHECKPOINT_DIR = Path(r"E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)
print("Checkpoint:", CHECKPOINT_DIR)
print("Seed:", RANDOM_SEED)


Train: E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_train.csv
Test : E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_test.csv
Checkpoint: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints
Seed: 42


# 2. Load the Actual Dataset and Verify Target Names

The supplied dataset uses `log_target` as the log-scale target and `target_usd` as the original USD target. Both are explicitly verified before the pipeline starts so the notebook cannot silently use a guessed target name.

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

TARGET_LOG = "log_target"
TARGET_USD = "target_usd"

assert TARGET_LOG in train_df.columns
assert TARGET_USD in train_df.columns
assert TARGET_LOG in test_df.columns
assert TARGET_USD in test_df.columns

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Log target :", TARGET_LOG)
print("USD target :", TARGET_USD)

print("\nTarget summary:")
display(train_df[[TARGET_LOG, TARGET_USD]].describe())


Train shape: (16000, 81)
Test shape : (4000, 81)
Log target : log_target
USD target : target_usd

Target summary:


,log_target,target_usd
count,16000.000000,1.600000e+04
mean,6.917456,1.695421e+04
std,3.048415,1.349467e+05
min,0.000000,0.000000e+00
25%,5.351536,2.099321e+02
50%,7.652783,2.105500e+03
75%,9.016543,8.237250e+03
max,16.199868,1.085209e+07


# 3. Remove Targets and Identify Feature Types

Only the two target columns are removed. All remaining columns are treated as candidate input features. The same feature schema is enforced for train and test.

In [3]:
EXCLUDED_TARGETS = {TARGET_LOG, TARGET_USD}

FEATURE_COLUMNS = [
    c for c in train_df.columns
    if c not in EXCLUDED_TARGETS
]

missing_in_test = sorted(set(FEATURE_COLUMNS) - set(test_df.columns))
if missing_in_test:
    raise ValueError(f"Test dataset is missing feature columns: {missing_in_test}")

X_all = train_df[FEATURE_COLUMNS].copy()
X_test_raw = test_df[FEATURE_COLUMNS].copy()

y_all_log = train_df[TARGET_LOG].to_numpy(dtype=np.float64)
y_test_log = test_df[TARGET_LOG].to_numpy(dtype=np.float64)

y_all_usd = train_df[TARGET_USD].to_numpy(dtype=np.float64)
y_test_usd = test_df[TARGET_USD].to_numpy(dtype=np.float64)

numeric_features = X_all.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [
    c for c in FEATURE_COLUMNS
    if c not in numeric_features
]

print("Total features:", len(FEATURE_COLUMNS))
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))


Total features: 79
Numeric: 79
Categorical: 0


# 4. Train/Validation Split

The original training dataset is split into a model-training portion and a validation portion. The validation set is used **only to select α**. The final test set remains untouched until the final metrics are calculated.

This is important because α is a model-selection parameter and should not be selected using the final test labels.

In [4]:
TRAIN_IDX, VAL_IDX = train_test_split(
    np.arange(len(X_all)),
    test_size=0.20,
    random_state=RANDOM_SEED,
    shuffle=True
)

X_train_raw = X_all.iloc[TRAIN_IDX].reset_index(drop=True)
X_val_raw = X_all.iloc[VAL_IDX].reset_index(drop=True)

y_train_log = y_all_log[TRAIN_IDX]
y_val_log = y_all_log[VAL_IDX]

y_train_usd = y_all_usd[TRAIN_IDX]
y_val_usd = y_all_usd[VAL_IDX]

print("Model-train:", len(TRAIN_IDX))
print("Validation :", len(VAL_IDX))
print("Test       :", len(test_df))


Model-train: 12800
Validation : 3200
Test       : 4000


# 5. Robust Preprocessing

The paper describes a robust preprocessing path involving categorical one-hot encoding, missing-value handling, robust scaling, and smoothing/clipping. This implementation uses scikit-learn imputation, one-hot encoding, and RobustScaler as the controlled preprocessing implementation.

The preprocessor is fitted on the model-training split only.

In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32
    ))
])

robust_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

X_train_orig = robust_preprocessor.fit_transform(X_train_raw)
X_val_orig = robust_preprocessor.transform(X_val_raw)
X_test_orig = robust_preprocessor.transform(X_test_raw)

X_train_orig = np.asarray(X_train_orig, dtype=np.float32)
X_val_orig = np.asarray(X_val_orig, dtype=np.float32)
X_test_orig = np.asarray(X_test_orig, dtype=np.float32)

print("Original processed train:", X_train_orig.shape)
print("Original processed val  :", X_val_orig.shape)
print("Original processed test :", X_test_orig.shape)


Original processed train: (12800, 79)
Original processed val  : (3200, 79)
Original processed test : (4000, 79)


# 6. Vanilla / Under-Tuned XGBoost GBDT

This is intentionally **not** the previously tuned XGBoost model.

The paper explicitly states that the GBDT component is kept under-tuned because its purpose is to generate informative sparse embeddings rather than maximize standalone predictive performance. citeturn0view0

The model is trained only on the model-training split. Its job here is to create leaf-routing information.

In [6]:
VANILLA_XGB_PARAMS = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}

vanilla_xgb = xgb.XGBRegressor(**VANILLA_XGB_PARAMS)

xgb_start = time.perf_counter()
vanilla_xgb.fit(X_train_orig, y_train_log)
xgb_training_time = time.perf_counter() - xgb_start

print("Vanilla XGBoost trained.")
print("Training time:", xgb_training_time)
print("Parameters:")
for k, v in VANILLA_XGB_PARAMS.items():
    print(f"  {k}: {v}")


Vanilla XGBoost trained.
Training time: 0.36144269999931566
Parameters:
  n_estimators: 100
  max_depth: 6
  learning_rate: 0.1
  subsample: 1.0
  colsample_bytree: 1.0
  min_child_weight: 1
  reg_lambda: 1.0
  reg_alpha: 0.0
  objective: reg:squarederror
  eval_metric: rmse
  tree_method: hist
  random_state: 42
  n_jobs: -1


# 7. Extract GBDT Leaf Indices

Every sample is mapped to one leaf in every XGBoost tree. This produces the raw leaf-index representation used to construct Γ(x).

In [7]:
leaf_train = vanilla_xgb.apply(X_train_orig)
leaf_val = vanilla_xgb.apply(X_val_orig)
leaf_test = vanilla_xgb.apply(X_test_orig)

leaf_train = np.asarray(leaf_train)
leaf_val = np.asarray(leaf_val)
leaf_test = np.asarray(leaf_test)

if leaf_train.ndim == 1:
    leaf_train = leaf_train[:, None]
    leaf_val = leaf_val[:, None]
    leaf_test = leaf_test[:, None]

print("Leaf train:", leaf_train.shape)
print("Leaf val  :", leaf_val.shape)
print("Leaf test :", leaf_test.shape)


Leaf train: (12800, 100)
Leaf val  : (3200, 100)
Leaf test : (4000, 100)


# 8. One-Hot GBDT Leaf Embedding Γ(x)

The paper defines Γ(x) by one-hot encoding the leaf reached in each tree and concatenating those one-hot vectors across the ensemble. This creates a sparse binary tree-derived embedding. citeturn0view0

In [8]:
leaf_train_df = pd.DataFrame(
    leaf_train,
    columns=[f"tree_{i}" for i in range(leaf_train.shape[1])]
)
leaf_val_df = pd.DataFrame(
    leaf_val,
    columns=leaf_train_df.columns
)
leaf_test_df = pd.DataFrame(
    leaf_test,
    columns=leaf_train_df.columns
)

leaf_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32
)

Gamma_train = leaf_encoder.fit_transform(leaf_train_df)
Gamma_val = leaf_encoder.transform(leaf_val_df)
Gamma_test = leaf_encoder.transform(leaf_test_df)

Gamma_train = np.asarray(Gamma_train, dtype=np.float32)
Gamma_val = np.asarray(Gamma_val, dtype=np.float32)
Gamma_test = np.asarray(Gamma_test, dtype=np.float32)

print("Γ(train):", Gamma_train.shape)
print("Γ(val)  :", Gamma_val.shape)
print("Γ(test) :", Gamma_test.shape)


Γ(train): (12800, 5483)
Γ(val)  : (3200, 5483)
Γ(test) : (4000, 5483)


# 9. Concatenate Original Features + Leaf Embedding

The paper explicitly considers concatenating the GBDT embedding with the robustly preprocessed original features. We freeze that choice for Method 2. citeturn0view0

In [9]:
Psi_train = np.hstack([X_train_orig, Gamma_train]).astype(np.float32)
Psi_val = np.hstack([X_val_orig, Gamma_val]).astype(np.float32)
Psi_test = np.hstack([X_test_orig, Gamma_test]).astype(np.float32)

print("Concatenated train:", Psi_train.shape)
print("Concatenated val  :", Psi_val.shape)
print("Concatenated test :", Psi_test.shape)


Concatenated train: (12800, 5562)
Concatenated val  : (3200, 5562)
Concatenated test : (4000, 5562)


# 10. Random Feature Expansion + ReLU

The paper's fixed-size embedding first applies a random linear projection followed by a pointwise nonlinearity such as ReLU. The random matrix is sampled once with a reproducible seed and is fitted/created from the training dimensionality only. citeturn0view0

In [10]:
RANDOM_FEATURES = 1024

input_dim = Psi_train.shape[1]

# Paper-inspired Gaussian random feature scale.
# Omega has shape [input_dim, RANDOM_FEATURES].
Omega = rng.normal(
    loc=0.0,
    scale=np.sqrt(2.0 / RANDOM_FEATURES),
    size=(input_dim, RANDOM_FEATURES)
).astype(np.float32)

def random_relu_features(X, Omega):
    Z = X @ Omega
    return np.maximum(Z, 0.0).astype(np.float32)

R_train = random_relu_features(Psi_train, Omega)
R_val = random_relu_features(Psi_val, Omega)
R_test = random_relu_features(Psi_test, Omega)

print("Random-ReLU train:", R_train.shape)
print("Random-ReLU val  :", R_val.shape)
print("Random-ReLU test :", R_test.shape)


Random-ReLU train: (12800, 1024)
Random-ReLU val  : (3200, 1024)
Random-ReLU test : (4000, 1024)


# 11. PCA to Fixed-Size Representation

PCA is applied **after** the randomized ReLU expansion, matching the paper's ordering. The PCA transformer is fitted only on the model-training representation. citeturn0view0

In [11]:
MAIN_DIM = 128

MAIN_DIM = min(
    MAIN_DIM,
    R_train.shape[1],
    R_train.shape[0] - 1
)

pca = PCA(
    n_components=MAIN_DIM,
    random_state=RANDOM_SEED
)

Z_train_pca = pca.fit_transform(R_train).astype(np.float32)
Z_val_pca = pca.transform(R_val).astype(np.float32)
Z_test_pca = pca.transform(R_test).astype(np.float32)

print("PCA dimension:", MAIN_DIM)
print("Explained variance:", pca.explained_variance_ratio_.sum())
print("PCA train:", Z_train_pca.shape)
print("PCA val  :", Z_val_pca.shape)
print("PCA test :", Z_test_pca.shape)


PCA dimension: 128
Explained variance: 0.9960847
PCA train: (12800, 128)
PCA val  : (3200, 128)
PCA test : (4000, 128)


# 12. Feature-Wise Normalization

The paper additionally normalizes each PCA feature using training statistics. Those training means and standard deviations are reused unchanged for validation and test representations. citeturn0view0

In [12]:
norm_mean = Z_train_pca.mean(axis=0)
norm_std = Z_train_pca.std(axis=0)

NORM_EPS = 1e-8

def normalize_representation(Z):
    return (
        (Z - norm_mean) /
        np.sqrt(norm_std ** 2 + NORM_EPS)
    ).astype(np.float32)

Z_train = normalize_representation(Z_train_pca)
Z_val = normalize_representation(Z_val_pca)
Z_test = normalize_representation(Z_test_pca)

print("Normalized representation shape:", Z_train.shape)
print("Mean abs:", np.mean(np.abs(Z_train.mean(axis=0))))
print("Mean std:", np.mean(Z_train.std(axis=0)))


Normalized representation shape: (12800, 128)
Mean abs: 1.4574981e-08
Mean std: 0.9999999


# 13. MLP Main Network

The MLP operates on the fixed-size representation produced by the embedding stage. For this regression adaptation, the MLP predicts `log_target`.

The penultimate hidden layer is deliberately retained because the paper uses the main network's penultimate representation as the retrieval space. citeturn0view0

In [13]:
from sklearn.neural_network import MLPRegressor

MLP_HIDDEN_LAYERS = (512, 512, 512)

mlp = MLPRegressor(
    hidden_layer_sizes=MLP_HIDDEN_LAYERS,
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=128,
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,
    random_state=RANDOM_SEED,
    verbose=True
)

mlp_start = time.perf_counter()
mlp.fit(Z_train, y_train_log)
mlp_training_time = time.perf_counter() - mlp_start

y_train_main_log = mlp.predict(Z_train)
y_val_main_log = mlp.predict(Z_val)
y_test_main_log = mlp.predict(Z_test)

print("MLP training time:", mlp_training_time)
print("MLP iterations:", mlp.n_iter_)


Iteration 1, loss = 3.99693153
Validation score: 0.319779
Iteration 2, loss = 2.49183367
Validation score: 0.394704
Iteration 3, loss = 2.14110434
Validation score: 0.411428
Iteration 4, loss = 1.80535257
Validation score: 0.382064
Iteration 5, loss = 1.46873397
Validation score: 0.366253
Iteration 6, loss = 1.08821909
Validation score: 0.313482
Iteration 7, loss = 0.85499364
Validation score: 0.285922
Iteration 8, loss = 0.59554291
Validation score: 0.290093
Iteration 9, loss = 0.34934228
Validation score: 0.261513
Iteration 10, loss = 0.23835930
Validation score: 0.280064
Iteration 11, loss = 0.14798995
Validation score: 0.287982
Iteration 12, loss = 0.11320512
Validation score: 0.286100
Iteration 13, loss = 0.09580543
Validation score: 0.287746
Iteration 14, loss = 0.08025401
Validation score: 0.277485
Iteration 15, loss = 0.07546075
Validation score: 0.294277
Iteration 16, loss = 0.07241630
Validation score: 0.292357
Iteration 17, loss = 0.07831939
Validation score: 0.289352
Iterat

# 14. Extract the MLP Penultimate Representation

For scikit-learn's MLPRegressor, the hidden-layer activations are accessible through the fitted network weights. We explicitly forward-propagate through every hidden layer and stop before the final output layer.

This gives the analogue of the paper's `f_main`, i.e. the penultimate representation used by retrieval. citeturn0view0

In [14]:
def relu(x):
    return np.maximum(x, 0.0)

def mlp_penultimate_transform(model, X):
    A = np.asarray(X, dtype=np.float32)

    # Every layer except the final output layer.
    for layer_idx in range(len(model.coefs_) - 1):
        A = A @ model.coefs_[layer_idx] + model.intercepts_[layer_idx]
        A = relu(A)

    return np.asarray(A, dtype=np.float32)

H_train = mlp_penultimate_transform(mlp, Z_train)
H_val = mlp_penultimate_transform(mlp, Z_val)
H_test = mlp_penultimate_transform(mlp, Z_test)

print("Penultimate train:", H_train.shape)
print("Penultimate val  :", H_val.shape)
print("Penultimate test :", H_test.shape)


Penultimate train: (12800, 512)
Penultimate val  : (3200, 512)
Penultimate test : (4000, 512)


# 15. Soft k-NN Retrieval from Training Context

Retrieval is performed in the **penultimate MLP representation**, not in the PCA representation and not in the final prediction space.

The paper computes cosine similarity, divides by a temperature τ, applies softmax, and aggregates context labels. For this regression adaptation, the retrieved value is the weighted average of `y_train_log`.

Only model-training samples are used as the retrieval context. Validation and test targets are never inserted into the retrieval index.

In [15]:
RETRIEVAL_K = 64
RETRIEVAL_TEMPERATURE = 0.10

def l2_normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, 1e-12)

H_train_norm = l2_normalize_rows(H_train).astype(np.float32)
H_val_norm = l2_normalize_rows(H_val).astype(np.float32)
H_test_norm = l2_normalize_rows(H_test).astype(np.float32)

# NearestNeighbors is used only to obtain the top-k cosine neighbors.
# The actual weights follow the paper's cosine-similarity + temperature softmax.
knn = NearestNeighbors(
    n_neighbors=min(RETRIEVAL_K, len(H_train_norm)),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)
knn.fit(H_train_norm)

def soft_retrieval(query_rep, context_rep, context_y, k, temperature):
    distances, indices = knn.kneighbors(
        query_rep,
        n_neighbors=min(k, len(context_rep))
    )

    # cosine distance = 1 - cosine similarity
    similarities = 1.0 - distances
    logits = similarities / temperature

    # numerically stable softmax
    logits = logits - logits.max(axis=1, keepdims=True)
    weights = np.exp(logits)
    weights /= weights.sum(axis=1, keepdims=True)

    retrieved_targets = context_y[indices]
    predictions = np.sum(weights * retrieved_targets, axis=1)

    return predictions, similarities, weights, indices

y_val_retrieval_log, val_similarities, val_weights, val_indices = soft_retrieval(
    H_val_norm,
    H_train_norm,
    y_train_log,
    RETRIEVAL_K,
    RETRIEVAL_TEMPERATURE
)

y_test_retrieval_log, test_similarities, test_weights, test_indices = soft_retrieval(
    H_test_norm,
    H_train_norm,
    y_train_log,
    RETRIEVAL_K,
    RETRIEVAL_TEMPERATURE
)

print("Validation retrieval predictions:", y_val_retrieval_log.shape)
print("Test retrieval predictions:", y_test_retrieval_log.shape)


Validation retrieval predictions: (3200,)
Test retrieval predictions: (4000,)


# 16. Validation-Selected α

The paper defines the retrieval contribution as:

`O = (1 - α) O_main + α O_retrieval`

For this regression adaptation, α is selected using **validation RMSE_log**, not test RMSE_log. The test set is not used for choosing α.

The grid is deliberately explicit and reproducible.

In [16]:
ALPHA_GRID = [
    0.0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
    1.0
]

alpha_validation_rows = []

for alpha in ALPHA_GRID:
    y_val_blended = (
        (1.0 - alpha) * y_val_main_log
        + alpha * y_val_retrieval_log
    )

    alpha_validation_rows.append({
        "alpha": alpha,
        "MAE_log": mean_absolute_error(y_val_log, y_val_blended),
        "MSE_log": mean_squared_error(y_val_log, y_val_blended),
        "RMSE_log": np.sqrt(mean_squared_error(y_val_log, y_val_blended)),
        "R2_log": r2_score(y_val_log, y_val_blended),
    })

alpha_validation_results = (
    pd.DataFrame(alpha_validation_rows)
    .sort_values("RMSE_log")
    .reset_index(drop=True)
)

best_alpha = float(alpha_validation_results.loc[0, "alpha"])

display(alpha_validation_results)

print("Validation-selected alpha:", best_alpha)


,alpha,MAE_log,MSE_log,RMSE_log,R2_log
0,0.1,1.744092,5.640607,2.374996,0.376278
1,0.0,1.778328,5.687112,2.384767,0.371135
2,0.2,1.722491,5.704368,2.388382,0.369227
3,0.3,1.717645,5.878393,2.424540,0.349984
4,0.4,1.728828,6.162684,2.482475,0.318548
5,0.5,1.758031,6.557240,2.560711,0.274919
6,0.6,1.804110,7.062061,2.657454,0.219097
7,0.7,1.867898,7.677148,2.770767,0.151083
8,0.8,1.948047,8.402499,2.898706,0.070875
9,0.9,2.043176,9.238116,3.039427,-0.021525


Validation-selected alpha: 0.1


# 17. Final Test Prediction

The selected α is now frozen. The final test prediction combines the MLP main prediction and the soft retrieval prediction. No test target was involved in training, retrieval, or α selection.

In [17]:
y_test_final_log = (
    (1.0 - best_alpha) * y_test_main_log
    + best_alpha * y_test_retrieval_log
)

# Inverse transform log_target to USD.
# Assumes log_target = log1p(target_usd), consistent with the dataset design.
y_test_pred_usd = np.maximum(
    np.expm1(y_test_final_log),
    0.0
)

print("Final alpha:", best_alpha)
print("Final log prediction shape:", y_test_final_log.shape)
print("Final USD prediction shape:", y_test_pred_usd.shape)


Final alpha: 0.1
Final log prediction shape: (4000,)
Final USD prediction shape: (4000,)


# 18. Final Metrics — Required Table

This is the primary evaluation cell.

The requested metrics are reported exactly as:
`MAE_log`, `MSE_log`, `RMSE_log`, `R2_log`, `MAE_USD`, `MSE_USD`, `RMSE_USD`, `R2_USD`, `RMSLE`, and `Training_Time_Seconds`.

`RMSE_log` is the main benchmark metric. The earlier XGBoost baseline of 2.2361 is reported separately for comparison.

In [18]:
# Log-space metrics
MAE_log = mean_absolute_error(y_test_log, y_test_final_log)
MSE_log = mean_squared_error(y_test_log, y_test_final_log)
RMSE_log = np.sqrt(MSE_log)
R2_log = r2_score(y_test_log, y_test_final_log)

# USD-space metrics
MAE_USD = mean_absolute_error(y_test_usd, y_test_pred_usd)
MSE_USD = mean_squared_error(y_test_usd, y_test_pred_usd)
RMSE_USD = np.sqrt(MSE_USD)
R2_USD = r2_score(y_test_usd, y_test_pred_usd)

# RMSLE
y_test_usd_safe = np.maximum(y_test_usd, 0.0)
y_test_pred_usd_safe = np.maximum(y_test_pred_usd, 0.0)

RMSLE = np.sqrt(
    mean_squared_log_error(
        y_test_usd_safe,
        y_test_pred_usd_safe
    )
)

# Requested training time includes the vanilla GBDT embedding generator
# and the MLP main-network training.
total_training_time = xgb_training_time + mlp_training_time

final_metrics = pd.DataFrame([{
    "MAE_log": MAE_log,
    "MSE_log": MSE_log,
    "RMSE_log": RMSE_log,
    "R2_log": R2_log,
    "MAE_USD": MAE_USD,
    "MSE_USD": MSE_USD,
    "RMSE_USD": RMSE_USD,
    "R2_USD": R2_USD,
    "RMSLE": RMSLE,
    "Training_Time_Seconds": total_training_time
}])

display(final_metrics)


,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
0,1.791952,5.833059,2.415173,0.379114,18695.259308,1.226537e+10,110749.144697,-0.039307,2.415173,80.536459


# 19. Main vs Retrieval vs Final Hybrid

This diagnostic comparison verifies whether retrieval actually contributes beyond the MLP and whether the selected hybrid improves the log-scale benchmark.

In [19]:
comparison_rows = []

for name, pred in [
    ("MLP Main", y_test_main_log),
    ("Soft Retrieval", y_test_retrieval_log),
    ("Final Hybrid", y_test_final_log),
]:
    comparison_rows.append({
        "Model": name,
        "MAE_log": mean_absolute_error(y_test_log, pred),
        "MSE_log": mean_squared_error(y_test_log, pred),
        "RMSE_log": np.sqrt(mean_squared_error(y_test_log, pred)),
        "R2_log": r2_score(y_test_log, pred),
    })

comparison_results = pd.DataFrame(comparison_rows)

display(comparison_results)

XGB_BASELINE_RMSE_LOG = 2.2361

print(f"Existing XGBoost baseline RMSE_log: {XGB_BASELINE_RMSE_LOG:.6f}")
print(f"Method 2 RMSE_log:                 {RMSE_log:.6f}")
print(
    f"Absolute improvement:              "
    f"{XGB_BASELINE_RMSE_LOG - RMSE_log:.6f}"
)
print(
    f"Relative improvement (%):          "
    f"{100 * (XGB_BASELINE_RMSE_LOG - RMSE_log) / XGB_BASELINE_RMSE_LOG:.2f}%"
)
print("Benchmark beaten:", RMSE_log < XGB_BASELINE_RMSE_LOG)


,Model,MAE_log,MSE_log,RMSE_log,R2_log
0,MLP Main,1.826854,5.893803,2.427716,0.372649
1,Soft Retrieval,2.215020,10.312479,3.211305,-0.097686
2,Final Hybrid,1.791952,5.833059,2.415173,0.379114


Existing XGBoost baseline RMSE_log: 2.236100
Method 2 RMSE_log:                 2.415173
Absolute improvement:              -0.179073
Relative improvement (%):          -8.01%
Benchmark beaten: False


# 20. Save Final Results and Artifacts

The complete experiment is saved to the requested checkpoint directory. This includes the final metrics, validation α search, test predictions, preprocessing objects, random projection, PCA, MLP, leaf encoder, and experiment metadata.

In [20]:
metrics_path = CHECKPOINT_DIR / "method2_final_metrics.csv"
alpha_path = CHECKPOINT_DIR / "method2_alpha_validation_results.csv"
comparison_path = CHECKPOINT_DIR / "method2_model_comparison.csv"
predictions_path = CHECKPOINT_DIR / "method2_test_predictions.csv"
artifacts_path = CHECKPOINT_DIR / "method2_artifacts.joblib"
metadata_path = CHECKPOINT_DIR / "method2_metadata.json"

# Final metrics
final_metrics.to_csv(metrics_path, index=False)

# Alpha validation
alpha_validation_results.to_csv(alpha_path, index=False)

# Model comparison
comparison_results.to_csv(comparison_path, index=False)

# Predictions
predictions = test_df.copy()
predictions["y_pred_log_main"] = y_test_main_log
predictions["y_pred_log_retrieval"] = y_test_retrieval_log
predictions["y_pred_log_final"] = y_test_final_log
predictions["y_pred_usd_final"] = y_test_pred_usd
predictions["selected_alpha"] = best_alpha
predictions.to_csv(predictions_path, index=False)

# Reusable fitted artifacts
joblib.dump({
    "vanilla_xgb": vanilla_xgb,
    "robust_preprocessor": robust_preprocessor,
    "leaf_encoder": leaf_encoder,
    "Omega": Omega,
    "pca": pca,
    "norm_mean": norm_mean,
    "norm_std": norm_std,
    "mlp": mlp,
    "feature_columns": FEATURE_COLUMNS,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "best_alpha": best_alpha,
    "retrieval_k": RETRIEVAL_K,
    "retrieval_temperature": RETRIEVAL_TEMPERATURE,
    "random_seed": RANDOM_SEED,
}, artifacts_path)

metadata = {
    "method": "Method 2: Vanilla XGBoost leaf embedding + random ReLU + PCA + normalization + MLP + soft retrieval",
    "target_log": TARGET_LOG,
    "target_usd": TARGET_USD,
    "random_seed": RANDOM_SEED,
    "vanilla_xgb_params": VANILLA_XGB_PARAMS,
    "random_features": int(RANDOM_FEATURES),
    "main_representation_dim": int(MAIN_DIM),
    "mlp_hidden_layers": list(MLP_HIDDEN_LAYERS),
    "retrieval_k": int(RETRIEVAL_K),
    "retrieval_temperature": float(RETRIEVAL_TEMPERATURE),
    "alpha_grid": ALPHA_GRID,
    "selected_alpha": best_alpha,
    "xgb_training_time_seconds": float(xgb_training_time),
    "mlp_training_time_seconds": float(mlp_training_time),
    "total_training_time_seconds": float(total_training_time),
    "benchmark_rmse_log": XGB_BASELINE_RMSE_LOG,
    "final_rmse_log": float(RMSE_log),
    "benchmark_beaten": bool(RMSE_log < XGB_BASELINE_RMSE_LOG),
    "retrieval_context": "model-training split only",
    "test_used_for_alpha_selection": False
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved artifacts:")
for p in [
    metrics_path,
    alpha_path,
    comparison_path,
    predictions_path,
    artifacts_path,
    metadata_path
]:
    print(" ", p)


Saved artifacts:
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints\method2_final_metrics.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints\method2_alpha_validation_results.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints\method2_model_comparison.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints\method2_test_predictions.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints\method2_artifacts.joblib
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_3\checkpoints\method2_metadata.json


# 21. Final Result Block

Run this cell at the end of the notebook to display the exact final metric table and the benchmark decision.

In [21]:
print("=" * 90)
print("METHOD 2 — FINAL TEST RESULTS")
print("=" * 90)

display(final_metrics)

print(f"Selected alpha: {best_alpha}")
print(f"RMSE_log:       {RMSE_log:.6f}")
print(f"Baseline:       {XGB_BASELINE_RMSE_LOG:.6f}")
print(f"Beats baseline: {RMSE_log < XGB_BASELINE_RMSE_LOG}")
print("=" * 90)


METHOD 2 — FINAL TEST RESULTS


,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
0,1.791952,5.833059,2.415173,0.379114,18695.259308,1.226537e+10,110749.144697,-0.039307,2.415173,80.536459


Selected alpha: 0.1
RMSE_log:       2.415173
Baseline:       2.236100
Beats baseline: False
